In [ ]:
from pyspark.sql import SparkSession
import boto3

# ------------------------------------
# 1. إنشاء Spark Session
spark = (
    SparkSession.builder
    .appName("Bronze Layer - Upload Check")
    .master("local[2]")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0",
            "org.apache.hadoop:hadoop-aws:3.4.2",
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.10.1"
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .config("spark.hadoop.fs.s3a.access.key", "@$@$$@$")
    .config("spark.hadoop.fs.s3a.secret.key", "@$@$$@$")
    .config("spark.hadoop.fs.s3a.path.style.access", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

# ----------------------------------------------------------
# 2. تعريف الـ files وأماكنها
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
BRONZE_BUCKET  = "nyc-flights-bronze"
REGION         = "us-east-1" 



# الملفات المحلية واسم كل واحدة على اس3 
files = [
    {"local_path": "/home/ahmed-refat/Desktop/NYC_delays.csv",  "s3_key": "NYC_delays.csv"},
    {"local_path": "/home/ahmed-refat/Desktop/flights & airports/Raw Data(batch)/weather.csv",     "s3_key": "weather.csv"},
    {"local_path": "/home/ahmed-refat/Desktop/flights & airports/Raw Data(batch)/World_Airports.csv",    "s3_key": "airports.csv"},
]

# ----------------------------------------------
# 3. التحقق من وجود الملف على اس 3 ورفعه لو مش موجود

# الكونكشن بتاع اس 3 
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

def file_exists_on_s3(bucket, key):
    """بتتحقق لو الملف موجود على اس 3 ولا لأ"""
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except:
        return False

def upload_if_not_exists(local_path, bucket, key):
    """بترفع الملف بس لو مش موجود على اس3"""
    if file_exists_on_s3(bucket, key):
        print(f" موجود بالفعل على S3: {key} - تم التخطي")
    else:
        print(f" جاري الرفع: {key}")
        s3_client.upload_file(local_path, bucket, key)
        print(f"تم الرفع بنجاح: {key}")

# ---------------------------------------------------
# 4. تنفيذ الرفع على الـ 3 ملفات
for file in files:
    upload_if_not_exists(file["local_path"], BRONZE_BUCKET, file["s3_key"])

print("\n Bronze Layer جاهز على S3")

In [ ]:
#task 2
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# -----------------------------------------------------------
# 1. إنشاء Spark Session
spark = SparkSession.builder \
    .appName("Flight Data Warehouse - Silver") \
    .master("local[2]") \
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.hadoop:hadoop-aws:3.4.2",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",  
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.10.1"
        ])
    ) \
    .config("spark.hadoop.fs.s3a.access.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.secret.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(" Spark Started:", spark.version)

# ----------------------------------------------------
# 2. تحميل البيانات الخام (Bronze) من Local
flights_df = spark.read.csv(
    "/home/ahmed-refat/Desktop/NYC_delays.csv",
    header=True,
    inferSchema=True
)

# ------------------------------------------------
# 3. تنظيف البيانات (Silver)

flights_clean = flights_df \
    .withColumn("FL_DATE", F.to_date(F.col("FL_DATE"), "M/d/yyyy hh:mm:ss a")) \
    .drop(
        "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID",
        "DEST_AIRPORT_SEQ_ID", "DEST_CITY_MARKET_ID",
        "ARR_DELAY_GROUP", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME",
        "DISTANCE_GROUP", "ARR_TIME_BLK", "DEP_DELAY_NEW",
        "ARR_DELAY_NEW", "FLIGHTS"
    )

# تحويل أنواع البيانات
flights_clean = flights_clean \
    .withColumn("CANCELLED", F.col("CANCELLED").cast("integer")) \
    .withColumn("DIVERTED", F.col("DIVERTED").cast("integer"))

# ملء القيم الفارغة
flights_clean = flights_clean \
    .fillna(0, subset=[
        "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY",
        "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
        "DEP_DELAY", "ARR_DELAY"
    ])

# أعمدة جديدة
flights_clean = flights_clean \
    .withColumn("IS_DEP_DELAYED", F.when(F.col("DEP_DELAY") > 0, 1).otherwise(0)) \
    .withColumn("IS_ARR_DELAYED", F.when(F.col("ARR_DELAY") > 0, 1).otherwise(0))

# حذف التكرار
flights_clean = flights_clean.dropDuplicates()

# تعديل الوقت
flights_clean = flights_clean \
    .withColumn("CRS_DEP_TIME", F.lpad(F.col("CRS_DEP_TIME").cast("string"), 4, "0")) \
    .withColumn("CRS_DEP_TIME", F.to_timestamp(F.col("CRS_DEP_TIME"), "HHmm")) \
    .withColumn("CRS_DEP_TIME", F.col("CRS_DEP_TIME").cast("string").substr(12, 5)) \
    .withColumn("CRS_ARR_TIME", F.lpad(F.col("CRS_ARR_TIME").cast("string"), 4, "0")) \
    .withColumn("CRS_ARR_TIME", F.to_timestamp(F.col("CRS_ARR_TIME"), "HHmm")) \
    .withColumn("CRS_ARR_TIME", F.col("CRS_ARR_TIME").cast("string").substr(12, 5))

# ------------------------------------------
# 4. عرض النتيجة
flights_clean.printSchema()
flights_clean.show(3)

# -----------------------------------------------
# 5. حفظ البيانات على S3 (Silver)

flights_clean.write \
    .mode("overwrite") \
    .parquet("s3a://nyc-flights-silver/flights/")

print(" تم حفظ flights على S3 Silver بنجاح")

In [ ]:
#task 3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# -----------------------------------------------------------
#إنشاء Spark Session
spark = SparkSession.builder \
    .appName("weather Data Warehouse - Silver") \
    .master("local[2]") \
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.hadoop:hadoop-aws:3.4.2",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",  
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.10.1"
        ])
    ) \
    .config("spark.hadoop.fs.s3a.access.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.secret.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(" Spark Started:", spark.version)

#------------------------------------------------------
# لود  الداتا 
weather_df = spark.read.csv("/home/ahmed-refat/Desktop/flights & airports/Raw Data(batch)/weather.csv", header=True, inferSchema=True)


#----------------------------------------------
#data clean 
weather_clean = weather_df \
    .drop("feelslike", "dew", "solarradiation", "solarenergy", 
          "uvindex", "icon", "stations", "name") \
    .fillna(0.0, subset=["precip", "snow", "snowdepth", "windgust", "precipprob"]) \
    .fillna("None", subset=["preciptype", "severerisk"]) \
    .withColumn("datetime", F.col("datetime").cast("timestamp")) \
    .withColumn("date", F.to_date(F.col("datetime"))) \
    .withColumn("hour", F.hour(F.col("datetime"))) \
    .dropDuplicates()

weather_clean.show(3)

# -----------------------------------------------
# . حفظ البيانات على S3 (Silver)

weather_clean.write \
    .mode("overwrite") \
    .parquet("s3a://nyc-flights-silver/weather/")

print(" تم حفظ weather على S3 Silver بنجاح")

In [ ]:
#task 4 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# -----------------------------------------------------------
#إنشاء Spark Session
spark = SparkSession.builder \
    .appName("airports Data Warehouse - Silver") \
    .master("local[2]") \
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.hadoop:hadoop-aws:3.4.2",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",  
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.10.1"
        ])
    ) \
    .config("spark.hadoop.fs.s3a.access.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.secret.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


#---------------------------------------------------------------------------------------
# لود أول داتا ست
airports_df = spark.read.csv("/home/ahmed-refat/Desktop/flights & airports/Raw Data(batch)/World_Airports.csv", header=True, inferSchema=True)

airports_clean = airports_df \
    .select(
        "iata_code", "name", "type", "municipality",
        "iso_country", "iso_region", "latitude_deg",
        "longitude_deg", "elevation_ft", "scheduled_service"
    ) \
    .filter(F.col("iata_code").isNotNull()) \
    .dropDuplicates()

# -----------------------------------------------
# . حفظ البيانات على S3 (Silver)

airports_clean.write \
    .mode("overwrite") \
    .parquet("s3a://nyc-flights-silver/airports/")

print(" تم حفظ airports على S3 Silver بنجاح")

26/04/28 04:24:03 ERROR HeartbeatBackground: heartbeat error - message=Authentication token has expired.  The user must authenticate again.
net.snowflake.client.jdbc.SnowflakeReauthenticationRequest: Authentication token has expired.  The user must authenticate again.
	at net.snowflake.client.jdbc.SnowflakeUtil.checkErrorAndThrowExceptionSub(SnowflakeUtil.java:141)
	at net.snowflake.client.jdbc.SnowflakeUtil.checkErrorAndThrowExceptionIncludingReauth(SnowflakeUtil.java:73)
	at net.snowflake.client.core.SessionUtil.tokenRequest(SessionUtil.java:982)
	at net.snowflake.client.core.SessionUtil.renewSession(SessionUtil.java:889)
	at net.snowflake.client.core.SFSession.renewSession(SFSession.java:681)
	at net.snowflake.client.core.SFSession.heartbeat(SFSession.java:874)
	at net.snowflake.client.core.HeartbeatBackground.run(HeartbeatBackground.java:192)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.run(Futur

In [ ]:
# task 5 (create dimintions and facts then load it to s3 (gold layer))
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.700 pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ------------------------------------------------------
# 1. إنشاء Spark Session
spark = SparkSession.builder \
    .appName("Gold Layer - NYC Flights Analysis") \
    .master("local[2]") \
    .config("spark.hadoop.fs.s3a.access.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.secret.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# -------------------------------------
# 2. تحميل البيانات من Silver (S3)
flights_df  = spark.read.parquet("s3a://nyc-flights-silver/flights/")
weather_df  = spark.read.parquet("s3a://nyc-flights-silver/weather/")
airports_df = spark.read.parquet("s3a://nyc-flights-silver/airports/")
print(" تم تحميل البيانات من Silver بنجاح")

# --------------------------------------------
# 3. بناء DIM_DATE
# بنسيب الـ key كـ BIGINT من غير cast لـ INT عشان نتجنب الـ overflow
dim_date = flights_df \
    .select(F.col("FL_DATE").alias("full_date")) \
    .distinct() \
    .withColumn("date_key", F.monotonically_increasing_id()) \
    .withColumn("day", F.dayofmonth("full_date")) \
    .withColumn("day_name", F.date_format("full_date", "EEEE")) \
    .withColumn("week_of_year", F.weekofyear("full_date")) \
    .withColumn("month", F.month("full_date")) \
    .withColumn("month_name", F.date_format("full_date", "MMMM")) \
    .withColumn("quarter", F.quarter("full_date")) \
    .withColumn("year", F.year("full_date")) \
    .withColumn("is_weekend", F.when(F.dayofweek("full_date").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("season", F.when(F.month("full_date").isin([12, 1, 2]), "Winter")
                           .when(F.month("full_date").isin([3, 4, 5]), "Spring")
                           .when(F.month("full_date").isin([6, 7, 8]), "Summer")
                           .otherwise("Fall")) \
    .withColumn("holiday_flag", F.lit(0))

# ----------------------------------------------------
# 4. بناء DIM_TIME
dim_time = flights_df \
    .select(F.col("CRS_DEP_TIME").alias("time_label")) \
    .distinct() \
    .withColumn("time_key", F.monotonically_increasing_id()) \
    .withColumn("hour", F.col("time_label").substr(1, 2).cast("integer")) \
    .withColumn("minute", F.col("time_label").substr(4, 2).cast("integer")) \
    .withColumn("part_of_day", F.when(F.col("hour").between(6, 11), "Morning")
                                 .when(F.col("hour").between(12, 17), "Afternoon")
                                 .when(F.col("hour").between(18, 21), "Evening")
                                 .otherwise("Night")) \
    .withColumn("time_slot_1h", F.concat(F.col("hour").cast("string"), F.lit(":00"))) \
    .withColumn("time_slot_3h", F.concat((F.col("hour") - (F.col("hour") % 3)).cast("string"), F.lit(":00"))) \
    .withColumn("peak_hour_flag", F.when(
        F.col("hour").between(7, 9) | F.col("hour").between(17, 19), 1).otherwise(0))

# ------------------------------------------------
# 5. بناء DIM_AIRPORT
dim_airport = airports_df \
    .withColumn("airport_key", F.monotonically_increasing_id()) \
    .withColumnRenamed("name", "airport_name") \
    .withColumnRenamed("municipality", "city") \
    .withColumnRenamed("iso_country", "country") \
    .withColumnRenamed("iso_region", "state") \
    .withColumnRenamed("type", "airport_type") \
    .select(
        "airport_key", "iata_code", "airport_name", "city",
        "state", "country", "latitude_deg", "longitude_deg",
        "elevation_ft", "airport_type"
    )

# -------------------------------------------------
# 6. بناء DIM_CARRIER
dim_carrier = flights_df \
    .select(F.col("OP_UNIQUE_CARRIER").alias("carrier_code")) \
    .distinct() \
    .withColumn("carrier_key", F.monotonically_increasing_id()) \
    .withColumn("carrier_name", F.lit(None).cast("string"))

# ---------------------------------------------------------
# 7. بناء DIM_WEATHER_CONDITION
dim_weather_condition = weather_df \
    .select("conditions", "preciptype", "severerisk") \
    .distinct() \
    .withColumn("weather_condition_key", F.monotonically_increasing_id()) \
    .withColumnRenamed("conditions", "weather_main") \
    .withColumn("weather_description", F.col("weather_main")) \
    .withColumnRenamed("preciptype", "precipitation_type") \
    .withColumnRenamed("severerisk", "severity_level")

# -------------------------------------------
# 8. بناء FACT_FLIGHT_DELAY
date_dim    = dim_date.select("date_key", "full_date")
time_dim    = dim_time.select("time_key", "time_label")
origin_dim  = dim_airport.select(
    F.col("airport_key").alias("origin_airport_key"),
    F.col("iata_code").alias("origin_iata")
)
dest_dim    = dim_airport.select(
    F.col("airport_key").alias("dest_airport_key"),
    F.col("iata_code").alias("dest_iata")
)
carrier_dim = dim_carrier.select("carrier_key", "carrier_code")

fact_flight_delay = flights_df \
    .join(date_dim,    flights_df["FL_DATE"]          == date_dim["full_date"],       "left") \
    .join(time_dim,    flights_df["CRS_DEP_TIME"]      == time_dim["time_label"],      "left") \
    .join(origin_dim,  flights_df["ORIGIN"]            == origin_dim["origin_iata"],   "left") \
    .join(dest_dim,    flights_df["DEST"]              == dest_dim["dest_iata"],       "left") \
    .join(carrier_dim, flights_df["OP_UNIQUE_CARRIER"] == carrier_dim["carrier_code"], "left") \
    .withColumn("flight_delay_key", F.monotonically_increasing_id()) \
    .withColumn("flight_number", F.col("OP_CARRIER_FL_NUM").cast("string")) \
    .withColumn("delay_bucket",
                F.when(F.col("ARR_DELAY") <= 0, "No Delay")
                 .when(F.col("ARR_DELAY").between(1, 15),   "Minor (1-15 min)")
                 .when(F.col("ARR_DELAY").between(16, 45),  "Moderate (16-45 min)")
                 .when(F.col("ARR_DELAY").between(46, 120), "Severe (46-120 min)")
                 .otherwise("Critical (>120 min)")) \
    .withColumn("is_weather_related_flag", F.when(F.col("WEATHER_DELAY") > 0, 1).otherwise(0)) \
    .select(
        "flight_delay_key", "date_key", "time_key",
        "origin_airport_key", "dest_airport_key", "carrier_key",
        "flight_number", F.col("FL_DATE").alias("fl_date"),
        F.col("DEP_DELAY").alias("dep_delay_minutes"),
        F.col("ARR_DELAY").alias("arr_delay_minutes"),
        F.col("CARRIER_DELAY").alias("carrier_delay_minutes"),
        F.col("WEATHER_DELAY").alias("weather_delay_minutes"),
        F.col("NAS_DELAY").alias("nas_delay_minutes"),
        F.col("SECURITY_DELAY").alias("security_delay_minutes"),
        F.col("LATE_AIRCRAFT_DELAY").alias("late_aircraft_delay_minutes"),
        F.col("CRS_ELAPSED_TIME").alias("scheduled_elapsed_time"),
        F.col("ACTUAL_ELAPSED_TIME").alias("actual_elapsed_time"),
        F.col("AIR_TIME").alias("air_time"),
        F.col("DISTANCE").alias("distance"),
        F.col("CANCELLED").alias("cancelled_flag"),
        F.col("DIVERTED").alias("diverted_flag"),
        F.col("IS_DEP_DELAYED").alias("is_dep_delayed_flag"),
        F.col("IS_ARR_DELAYED").alias("is_arr_delayed_flag"),
        "is_weather_related_flag", "delay_bucket"
    )

# ---------------------------------------------------------------
# 9. بناء FACT_WEATHER_OBSERVATION
weather_date_dim = dim_date.select("date_key", "full_date")
weather_time_dim = dim_time.select("time_key", "time_label")
weather_cond_dim = dim_weather_condition.select("weather_condition_key", "weather_main")

fact_weather_observation = weather_df \
    .join(weather_date_dim, weather_df["date"]      == weather_date_dim["full_date"],     "left") \
    .join(weather_time_dim, weather_df["hour"]       == F.col("time_label").substr(1, 2).cast("integer"), "left") \
    .join(weather_cond_dim, weather_df["conditions"] == weather_cond_dim["weather_main"], "left") \
    .withColumn("weather_observation_key", F.monotonically_increasing_id()) \
    .withColumn("airport_key", F.lit(None).cast("long")) \
    .withColumn("weather_severity_score",
                F.when(F.col("severerisk") == "None",     0)
                 .when(F.col("severerisk") == "Low",      1)
                 .when(F.col("severerisk") == "Moderate", 2)
                 .when(F.col("severerisk") == "High",     3)
                 .otherwise(0)) \
    .select(
        "weather_observation_key", "date_key", "time_key",
        "airport_key", "weather_condition_key",
        F.col("temp").alias("temperature"), "humidity",
        F.col("precip").alias("precipitation"), "snow",
        F.col("windspeed").alias("wind_speed"),
        F.col("winddir").alias("wind_direction"),
        "visibility", F.col("sealevelpressure").alias("pressure"),
        "cloudcover", "weather_severity_score"
    )

# ------------------------------------------------
# 10. حفظ الـ Gold Layer على S3
dim_date.write.mode("overwrite").parquet("s3a://nyc-flights-gold/dim_date/")
dim_time.write.mode("overwrite").parquet("s3a://nyc-flights-gold/dim_time/")
dim_airport.write.mode("overwrite").parquet("s3a://nyc-flights-gold/dim_airport/")
dim_carrier.write.mode("overwrite").parquet("s3a://nyc-flights-gold/dim_carrier/")
dim_weather_condition.write.mode("overwrite").parquet("s3a://nyc-flights-gold/dim_weather_condition/")
fact_flight_delay.write.mode("overwrite").parquet("s3a://nyc-flights-gold/fact_flight_delay/")
fact_weather_observation.write.mode("overwrite").parquet("s3a://nyc-flights-gold/fact_weather_observation/")

print(" Gold Layer تم حفظه بنجاح على S3")

 تم تحميل البيانات من Silver بنجاح


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/home/ahmed-refat/miniconda3/envs/spark/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# task 6 (load data to snowflake)
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.700,net.snowflake:snowflake-jdbc:3.13.33,net.snowflake:spark-snowflake_2.13:2.14.0-spark_3.4 pyspark-shell'

from pyspark.sql import SparkSession

# -----------------------------------------------
# 1. إنشاء Spark Session
spark = SparkSession.builder \
    .appName("Gold to Snowflake") \
    .master("local[2]") \
    .config("spark.hadoop.fs.s3a.access.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.secret.key", "@$@$$@$") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ------------------------------------------------
# 2. Snowflake Connection Options
snowflake_options = {
    "sfURL":       "vk55508.eu-central-2.aws.snowflakecomputing.com",
    "sfUser":      "",
    "sfPassword":  "",
    "sfDatabase":  "NYC_FLIGHTS_DW",
    "sfSchema":    "PUBLIC",
    "sfWarehouse": "COMPUTE_WH" ,

}

# ----------------------------------------
# 3. تحميل البيانات من Gold (S3)
dim_date              = spark.read.parquet("s3a://nyc-flights-gold/dim_date/")
dim_time              = spark.read.parquet("s3a://nyc-flights-gold/dim_time/")
dim_airport           = spark.read.parquet("s3a://nyc-flights-gold/dim_airport/")
dim_carrier           = spark.read.parquet("s3a://nyc-flights-gold/dim_carrier/")
dim_weather_condition = spark.read.parquet("s3a://nyc-flights-gold/dim_weather_condition/")
fact_flight_delay     = spark.read.parquet("s3a://nyc-flights-gold/fact_flight_delay/")
fact_weather_obs      = spark.read.parquet("s3a://nyc-flights-gold/fact_weather_observation/")

print(" تم تحميل البيانات من Gold S3")

# ------------------------------------
# 4. helper function للرفع على Snowflake
def write_to_snowflake(df, table_name):
    df.write \
        .format("net.snowflake.spark.snowflake") \
        .options(**snowflake_options) \
        .option("dbtable", table_name) \
        .mode("overwrite") \
        .save()
    print(f" تم رفع {table_name} على Snowflake")

# ---------------------------------------------------------
# 5. رفع الـ Dimensions أولاً عشان الـ FK يشتغل
write_to_snowflake(dim_date,              "DIM_DATE")
write_to_snowflake(dim_time,              "DIM_TIME")
write_to_snowflake(dim_airport,           "DIM_AIRPORT")
write_to_snowflake(dim_carrier,           "DIM_CARRIER")
write_to_snowflake(dim_weather_condition, "DIM_WEATHER_CONDITION")

# ----------------------------------------------
# 6. رفع الـ Facts بعد الـ Dimensions
write_to_snowflake(fact_flight_delay, "FACT_FLIGHT_DELAY")
write_to_snowflake(fact_weather_obs,  "FACT_WEATHER_OBSERVATION")

print(" تم رفع كل البيانات على Snowflake بنجاح")

 تم تحميل البيانات من Gold S3


26/04/27 23:00:01 WARN SnowflakeConnectorUtils$: Query pushdown is not supported because you are using Spark 4.1.0 with a connector designed to support Spark 3.4. Either use the version of Spark supported by the connector or install a version of the connector that supports your version of Spark.
26/04/27 23:00:10 WARN SnowflakeConnectorUtils$: Query pushdown is not supported because you are using Spark 4.1.0 with a connector designed to support Spark 3.4. Either use the version of Spark supported by the connector or install a version of the connector that supports your version of Spark.


 تم رفع DIM_DATE على Snowflake


26/04/27 23:00:10 WARN SnowflakeConnectorUtils$: Query pushdown is not supported because you are using Spark 4.1.0 with a connector designed to support Spark 3.4. Either use the version of Spark supported by the connector or install a version of the connector that supports your version of Spark.
26/04/27 23:00:15 WARN SnowflakeConnectorUtils$: Query pushdown is not supported because you are using Spark 4.1.0 with a connector designed to support Spark 3.4. Either use the version of Spark supported by the connector or install a version of the connector that supports your version of Spark.


 تم رفع DIM_TIME على Snowflake


...

 تم رفع DIM_WEATHER_CONDITION على Snowflake


26/04/27 23:00:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/27 23:01:33 WARN SnowflakeConnectorUtils$: Query pushdown is not supported because you are using Spark 4.1.0 with a connector designed to support Spark 3.4. Either use the version of Spark supported by the connector or install a version of the connector that supports your version of Spark.
26/04/27 23:01:33 WARN SnowflakeConnectorUtils$: Query pushdown is not supported because you are using Spark 4.1.0 with a connector designed to support Spark 3.4. Either use the version of Spark supported by the connector or install a version of the connector that supports your version of Spark.


 تم رفع FACT_FLIGHT_DELAY على Snowflake


 تم رفع FACT_WEATHER_OBSERVATION على Snowflake
 تم رفع كل البيانات على Snowflake بنجاح


26/04/27 23:02:00 WARN SnowflakeConnectorUtils$: Query pushdown is not supported because you are using Spark 4.1.0 with a connector designed to support Spark 3.4. Either use the version of Spark supported by the connector or install a version of the connector that supports your version of Spark.


In [ ]:
from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.utils.dates import days_ago
from datetime import timedelta

# --------------------------------------
# Default Arguments
default_args = {
    "owner": "ahmed-refat",
    "depends_on_past": False,
    "email_on_failure": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

# ----------------------------------------------
# DAG Definition
with DAG(
    dag_id="nyc_flights_data_warehouse",
    default_args=default_args,
    description="NYC Flights Data Warehouse Pipeline",
    schedule_interval="@once",
    start_date=days_ago(1),
    catchup=False,
    tags=["nyc", "flights", "data-warehouse"],
) as dag:

    # Task 1: رفع الداتا على S3 Bronze لو مش موجودة
    upload_bronze = BashOperator(
        task_id="upload_bronze",
        bash_command="python /home/ahmed-refat/dags/scripts/01_upload_bronze.py",
    )

    # Task 2 و 3 و 4: تنظيف الـ 3 datasets بالتوازي
    process_flights = BashOperator(
        task_id="process_flights",
        bash_command="python /home/ahmed-refat/dags/scripts/02_process_flights.py",
    )

    process_weather = BashOperator(
        task_id="process_weather",
        bash_command="python /home/ahmed-refat/dags/scripts/03_process_weather.py",
    )

    process_airports = BashOperator(
        task_id="process_airports",
        bash_command="python /home/ahmed-refat/dags/scripts/04_process_airports.py",
    )

    # Task 5: بناء Gold Layer بعد ما الـ 3 tasks خلصوا
    build_gold = BashOperator(
        task_id="build_gold",
        bash_command="python /home/ahmed-refat/dags/scripts/05_build_gold.py",
    )

    # Task 6: رفع الداتا على Snowflake
    upload_snowflake = BashOperator(
        task_id="upload_snowflake",
        bash_command="python /home/ahmed-refat/dags/scripts/06_upload_snowflake.py",
    )

    # ==============================
    # Task Dependencies
    # التوازي: الـ 3 processing tasks بيشتغلوا مع بعض بعد الـ upload
    upload_bronze >> [process_flights, process_weather, process_airports] >> build_gold >> upload_snowflake

--- System Versions Info ---
Python Version: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
Spark Version: 4.1.0
Java Version: 17.0.18
Hadoop Version: 3.4.2
----------------------------
